# SkyRisk Lab: reproducción externa R2 del entorno Colab

Esta revisión no pertenece al snapshot original ni lo modifica. Cargue el ZIP
original con su SHA-256 original. No lo regenere a partir del trabajo actual.
Use un runtime alojado de Colab nuevo (CPU), no un runtime local.
La preparación evita ensurepip mediante venv --without-pip y dirige el pip del
intérprete principal al venv con --python. No cambia las seis pruebas ni límites.
Se entrega sin ejecutar: no contiene resultados nuevos ni sustituye evidencias.


In [ ]:
# Copiar el valor sha256_zip que imprimió preparar_colab.py en el equipo local.
SHA256_ESPERADO = "PEGAR_AQUI_EL_SHA256_DEL_ZIP"
import re
if not re.fullmatch(r"[0-9a-f]{64}", SHA256_ESPERADO):
    raise ValueError("Pegue primero el SHA-256 real del paquete local")


## 1. Subir y verificar los bytes antes de ejecutar código del paquete

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, zipfile
subidos = files.upload()
if len(subidos) != 1:
    raise ValueError("Seleccione únicamente el ZIP de fuentes")
bruto = next(iter(subidos.values()))
if len(bruto) > 20 * 1024**2 or hashlib.sha256(bruto).hexdigest() != SHA256_ESPERADO:
    raise ValueError("El ZIP no coincide con la huella local")
with zipfile.ZipFile(io.BytesIO(bruto)) as z:
    infos = z.infolist()
    nombres = [i.filename for i in infos]
    if len(nombres) != len(set(nombres)) or len(nombres) > 128 or sum(i.file_size for i in infos) > 20 * 1024**2:
        raise ValueError("Miembros duplicados o paquete excesivo")
    for i in infos:
        p = PurePosixPath(i.filename)
        if p.is_absolute() or '..' in p.parts or chr(92) in i.filename or ':' in i.filename or i.is_dir() or (i.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Ruta o tipo ZIP no permitido")
    manifiesto = json.loads(z.read('MANIFIESTO_COLAB.json'))
    identidad = hashlib.sha256(json.dumps(manifiesto['archivos'], sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    if manifiesto['formato'] != 1 or manifiesto['rama'] != 'desarrollo/ampliacion-academica' or identidad != manifiesto['snapshot_id'] or set(nombres) != set(manifiesto['archivos']) | {'MANIFIESTO_COLAB.json'}:
        raise ValueError("Manifiesto inconsistente")
    for nombre, registro in manifiesto['archivos'].items():
        contenido = z.read(nombre)
        if len(contenido) != registro['bytes'] or hashlib.sha256(contenido).hexdigest() != registro['sha256']:
            raise ValueError("Contenido alterado: " + nombre)
    RAIZ = Path('/content') / ('skyrisk_' + identidad[:12])
    RAIZ.mkdir(exist_ok=False)  # No sobrescribir una extracción anterior.
    for nombre in nombres:
        ruta = RAIZ / nombre
        ruta.parent.mkdir(parents=True, exist_ok=True)
        ruta.write_bytes(z.read(nombre))
print('Snapshot verificado:', identidad)
print('Commit base (no identifica los cambios locales):', manifiesto['revision_base'])


## 2. Entorno externo R2 sin ensurepip
Requiere Python >=3.12 y pip principal con soporte de --python. No actualizar ni
instalar pip globalmente si falta soporte: detenerse y conservar el diagnóstico.
Se usan nombres R2 y se rechazan entornos o informes ya existentes.


In [ ]:
import subprocess, sys
if sys.version_info < (3, 12):
    raise RuntimeError('Este proyecto necesita Python >=3.12')
VENV = Path('/content') / ('venv_skyrisk_r2_' + identidad[:12])
EVIDENCIA = Path('/content') / ('evidencia_skyrisk_r2_' + identidad[:12])
for directorio in (VENV, EVIDENCIA):
    if directorio.exists() or directorio.is_symlink():
        raise FileExistsError('No sobrescribir entorno o evidencia: ' + str(directorio))
PYTHON = VENV / 'bin' / 'python'
pins = manifiesto['dependencias_cpsat']
requisitos = (RAIZ / 'requirements.txt').read_text(encoding='utf-8-sig').splitlines()
if any(pin not in requisitos for pin in pins):
    raise ValueError('Los pins mínimos difieren del proyecto')
# Verificar soporte antes de crear un entorno. No usar ensurepip ni get-pip.
subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON), '--help'], check=True)
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PIP_VENV = [sys.executable, '-m', 'pip', '--python', str(PYTHON)]
subprocess.run([*PIP_VENV, 'install', '--only-binary=:all:', *pins], check=True)
subprocess.run([*PIP_VENV, 'check'], check=True)
# Comprobar versiones y ubicación usando el intérprete del entorno aislado.
verificar_instalacion = """import importlib.metadata as m, json, sys
from pathlib import Path
base = Path(sys.prefix).resolve()
if sys.prefix == sys.base_prefix:
    raise RuntimeError('No se está usando el entorno virtual')
registros = {}
for pin in json.loads(sys.argv[1]):
    nombre, version = pin.split('==')
    dist = m.distribution(nombre)
    ubicacion = Path(dist.locate_file('')).resolve()
    if dist.version != version or not ubicacion.is_relative_to(base):
        raise RuntimeError('Dependencia fuera del entorno o versión incorrecta: ' + nombre)
    registros[nombre] = {'version': dist.version, 'ubicacion': str(ubicacion)}
print(json.dumps(registros, indent=2))
"""
subprocess.run([str(PYTHON), '-B', '-c', verificar_instalacion, json.dumps(pins)], check=True)
EVIDENCIA.mkdir(exist_ok=False)
(EVIDENCIA / 'MANIFIESTO_COLAB.json').write_bytes((RAIZ / 'MANIFIESTO_COLAB.json').read_bytes())
freeze = subprocess.check_output([*PIP_VENV, 'freeze'])
(EVIDENCIA / 'requirements_instalados.txt').write_bytes(freeze)
RUNNER = RAIZ / 'herramientas/validadores/validar_cpsat_colab.py'


## 3. Memoria y cgroup: preflight sin ejecutar CP-SAT
No parchear la admisión ni reducir el mínimo de 2 GiB.

In [ ]:
preflight = subprocess.run([str(PYTHON), '-B', str(RUNNER), '--salida', str(EVIDENCIA / 'preflight')], cwd=RAIZ)
informe_preflight = json.loads((EVIDENCIA / 'preflight/resultado.json').read_text())
print(json.dumps(informe_preflight, indent=2, ensure_ascii=False))
PREFLIGHT_OK = preflight.returncode == 0 and informe_preflight['estado'] == 'PREFLIGHT_LISTO_NO_EJECUTADO'
if not PREFLIGHT_OK:
    raise RuntimeError('Admisión rechazada: conservar informe y no ejecutar la siguiente celda')


## 4. Ejecutar exactamente las seis pruebas, secuencialmente
Solo esta celda autoriza ejecución real. Se mantienen las verificaciones internas del proyecto.

In [ ]:
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight no autorizado')
resultado = subprocess.run([str(PYTHON), '-B', str(RUNNER), '--ejecutar', '--salida', str(EVIDENCIA / 'ejecucion')], cwd=RAIZ)
informe = json.loads((EVIDENCIA / 'ejecucion/resultado.json').read_text())
print(json.dumps(informe, indent=2, ensure_ascii=False))
print((EVIDENCIA / 'ejecucion/unittest.log').read_text())
if resultado.returncode != 0 or not informe['seis_aprobadas']:
    print('VALIDACIÓN NO COMPLETA: descargar evidencia para revisión; no declarar seis aprobadas.')


## 5. Descargar evidencia incluso si hubo rechazo o fallos
Conservar el archivo antes de desconectar el runtime. No sustituir la evidencia histórica original.

In [ ]:
import shutil
ZIP_EVIDENCIA = Path('/content') / ('resultados_skyrisk_' + identidad[:12] + '.zip')
if ZIP_EVIDENCIA.exists():
    raise FileExistsError('No sobrescribir una descarga anterior')
shutil.make_archive(str(ZIP_EVIDENCIA.with_suffix('')), 'zip', EVIDENCIA)
print('SHA-256 de evidencia:', hashlib.sha256(ZIP_EVIDENCIA.read_bytes()).hexdigest())
files.download(str(ZIP_EVIDENCIA))
